Import des données

In [23]:
import pandas as pd
df = pd.read_csv(
 'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb' 
)

/tmp/ipykernel_9918/3470812471.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


1 Exploration générale

Question 1

In [24]:
# Filtrer les données sans code département
df = df[df["code_departement"] != "fr_etranger"]

# Ajouter des zéros devant l'ancien code commune
df["code_commune"] = df["code_commune"].astype(str).str.zfill(3)

# Coller le code département et l'ancien code commune
df["code_commune"] = df["code_departement"] + df["code_commune"]

# Nouvelle colonne prénom NOM
df["candidat"] =  df["prenom"] + " " + df["nom"]
print(df)

       code_departement       libelle_departement code_commune  \
0                    01                       Ain        01001   
1                    01                       Ain        01002   
2                    01                       Ain        01004   
3                    01                       Ain        01005   
4                    01                       Ain        01006   
...                 ...                       ...          ...   
528460              975  Saint-Pierre-et-Miquelon       975501   
528461              975  Saint-Pierre-et-Miquelon       975502   
528462              986          Wallis et Futuna       986001   
528463              977          Saint-Barthélémy       977701   
528464              978              Saint-Martin       978801   

                libelle_commune    prenom      nom  voix          candidat  
0       L'Abergement-Clémenciat  Nathalie  ARTHAUD     3  Nathalie ARTHAUD  
1         L'Abergement-de-Varey  Nathalie  ARTHAUD   

Question 2

In [25]:
# Liste des candidats sans votes nuls
liste_candidat = df["candidat"].dropna().unique()

# Nombre candidats
candidat = len(liste_candidat)

# Affichage du nombre de candidats
print(f"En 2022, il y avait {candidat} candidats à l'élection présidentielle.")

En 2022, il y avait 12 candidats à l'élection présidentielle.


Question 3

In [26]:
score_national = (df.groupby("candidat").agg({'voix': sum}).sort_values('voix', ascending = False))

score_national["Score (% votes exprimés)"] = score_national["voix"] / score_national["voix"].sum() * 100
score_national.rename(columns={'voix': 'Nombre votes (total)'}, inplace=True)   

print(score_national)

                       Nombre votes (total)  Score (% votes exprimés)
candidat                                                             
Emmanuel MACRON                     9558101                 27.597454
Marine LE PEN                       8107448                 23.408930
Jean-Luc MÉLENCHON                  7603126                 21.952783
Éric ZEMMOUR                        2441974                  7.050801
Valérie PÉCRESSE                    1658045                  4.787334
Yannick JADOT                       1587079                  4.582431
Jean LASSALLE                       1095423                  3.162855
Fabien ROUSSEL                       799156                  2.307432
Nicolas DUPONT-AIGNAN                718102                  2.073402
Anne HIDALGO                         603989                  1.743919
Philippe POUTOU                      265759                  0.767336
Nathalie ARTHAUD                     195794                  0.565323


/tmp/ipykernel_9918/3016939936.py:1: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  score_national = (df.groupby("candidat").agg({'voix': sum}).sort_values('voix', ascending = False))


Question 4

In [27]:
# Calcul du nombre de voix des candidats pour chaque département
score_departements = (
    df.groupby(["candidat", "code_departement"])
    .agg({'voix': 'sum'})
    .reset_index()
    .sort_values(
        by=['code_departement', 'voix'], # trie par département et nombre de voix
        ascending=[True, False]
    )
    )

# Calcul du score
score_departements["Score (% votes exprimés)"] = round(
    (
    (score_departements["voix"] /
     score_departements.groupby("code_departement")["voix"]
     .transform("sum") #  somme pour chacun des candidats de chacun des départements
     ) * 100
    ), 2
)

score_departements.rename(columns={'voix': 'votes'}, inplace=True)

Test pour le département de l'Aude (11)

In [28]:
score_departement_11 = score_departements[score_departements["code_departement"] == '11']
print(score_departement_11)

                   candidat code_departement  votes  Score (% votes exprimés)
545           Marine LE PEN               11  64027                     30.14
117         Emmanuel MACRON               11  43104                     20.29
438      Jean-Luc MÉLENCHON               11  42039                     19.79
1187           Éric ZEMMOUR               11  18434                      8.68
331           Jean LASSALLE               11  12382                      5.83
973        Valérie PÉCRESSE               11   7350                      3.46
1080          Yannick JADOT               11   6322                      2.98
10             Anne HIDALGO               11   6166                      2.90
224          Fabien ROUSSEL               11   5622                      2.65
759   Nicolas DUPONT-AIGNAN               11   4206                      1.98
866         Philippe POUTOU               11   1748                      0.82
652        Nathalie ARTHAUD               11   1026             

Question 5

In [29]:
# Renommer les variables
score_national.rename(columns={
    "Score (% votes exprimés)": "score_national"
}, inplace=True)
score_national.rename(columns={
    "Nombre votes (total)": "vote_national"
}, inplace=True)

score_departements.rename(columns={
    "votes": "vote_departement"
}, inplace=True)
score_departements.rename(columns={
    "Score (% votes exprimés)": "score_departement"
}, inplace=True)

# Jointure
score_joint = score_departements.merge(
    score_national,
    on="candidat",
    how="left"
)

Test pour le département de l'Aude (11)

In [30]:
score_joint_11 = score_joint[
    score_joint["code_departement"] == "11"
]

print(score_joint_11)

                  candidat code_departement  vote_departement  \
120          Marine LE PEN               11             64027   
121        Emmanuel MACRON               11             43104   
122     Jean-Luc MÉLENCHON               11             42039   
123           Éric ZEMMOUR               11             18434   
124          Jean LASSALLE               11             12382   
125       Valérie PÉCRESSE               11              7350   
126          Yannick JADOT               11              6322   
127           Anne HIDALGO               11              6166   
128         Fabien ROUSSEL               11              5622   
129  Nicolas DUPONT-AIGNAN               11              4206   
130        Philippe POUTOU               11              1748   
131       Nathalie ARTHAUD               11              1026   

     score_departement  vote_national  score_national  
120              30.14        8107448       23.408930  
121              20.29        9558101     

Question 6

In [33]:
score_joint["surrepresentation (%)"] = round(
    (
        score_joint["score_departement"] /
        score_joint["score_national"] - 1
    ) * 100,
    2
)
print(score_joint)

                candidat code_departement  vote_departement  \
0        Emmanuel MACRON               01             92206   
1          Marine LE PEN               01             86755   
2     Jean-Luc MÉLENCHON               01             57832   
3           Éric ZEMMOUR               01             27530   
4       Valérie PÉCRESSE               01             17572   
...                  ...              ...               ...   
1279       Jean LASSALLE              988              1031   
1280        Anne HIDALGO              988               963   
1281    Nathalie ARTHAUD              988               565   
1282     Philippe POUTOU              988               560   
1283      Fabien ROUSSEL              988               399   

      score_departement  vote_national  score_national  surrepresentation (%)  
0                 27.69        9558101       27.597454                   0.34  
1                 26.05        8107448       23.408930                  11.28  
2  